In [140]:
import numpy as np
import numpy.linalg as lin
import qutip as qt
import math

In [141]:
def sqrt(A):
    """Takes the square root of a matrix

    Parameters
    ----------
    A : matrix

    Returns
    -------
    matrix
        a matrix B such that B*B = A
    """
    eigvals, eigvecs = lin.eig(A)
    eigvals = np.sqrt(eigvals)
    return np.matmul(np.matmul(eigvecs, np.diag(eigvals)), lin.inv(eigvecs))

def log(A):
    """Takes the natural log of a matrix

    Parameters
    ----------
    A : matrix

    Returns
    -------
    matrix
        ln(A)
    """
    eigvals, eigvecs = lin.eig(A)
    eigvals = np.log(eigvals)
    return np.matmul(np.matmul(eigvecs, np.diag(eigvals)), lin.inv(eigvecs))

def exp(A):
    """Takes the exponential of a matrix

    Parameters
    ----------
    A : matrix

    Returns
    -------
    matrix
        e^A
    """
    eigvals, eigvecs = lin.eig(A)
    eigvals = np.exp(eigvals)
    return np.matmul(np.matmul(eigvecs, np.diag(eigvals)), lin.inv(eigvecs))

In [142]:
def tr_x(p_xyz, dx, dy, dz):
    """Traces system x out of p_xyz via partial trace

    Parameters
    ----------
    p_xyz : matrix
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        p_yz = tr_x(p_xyz)
    """
    return np.trace(p_xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=0, axis2=3)

def tr_y(p_xyz, dx, dy, dz):
    """Traces system y out of p_xyz via partial trace

    Parameters
    ----------
    p_xyz : matrix
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        p_xz = tr_y(p_xyz)
    """
    return np.trace(p_xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=1, axis2=4)

def tr_z(p_xyz, dx, dy, dz):
    """Traces system z out of p_xyz via partial trace

    Parameters
    ----------
    p_xyz : matrix
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        p_xy = tr_z(p_xyz)
    """
    return np.trace(p_xyz.reshape(dx, dy, dz, dx, dy, dz), axis1=2, axis2=5)

def tr_yz(p_xyz, dx, dy, dz):
    """Traces system y and z out of p_xyz via partial trace

    Parameters
    ----------
    p_xyz : matrix
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        p_x = tr_yz(p_xyz)
    """
    p_xy = tr_z(p_xyz, dx, dy, dz).reshape(dx*dy, dx*dy)
    return np.trace(p_xy.reshape(dx, dy, dx, dy), axis1=1, axis2=3)

def tr_xz(p_xyz, dx, dy, dz):
    """Traces system x and z out of p_xyz via partial trace

    Parameters
    ----------
    p_xyz : matrix
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        p_y = tr_xz(p_xyz)
    """
    p_xy = tr_z(p_xyz, dx, dy, dz).reshape(dx*dy, dx*dy)
    return np.trace(p_xy.reshape(dx, dy, dx, dy), axis1=0, axis2=2)

def tr_xy(p_xyz, dx, dy, dz):
    """Traces system x and y out of p_xyz via partial trace

    Parameters
    ----------
    p_xyz : matrix
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        p_z = tr_xy(p_xyz)
    """
    p_xz = tr_y(p_xyz, dx, dy, dz)
    return np.trace(p_xz.reshape(dx, dz, dx, dz), axis1=0, axis2 = 2)

In [143]:
def get_zlx(p_xyz, dx, dy, dz):
    """Calculates the conditional state of z given x (p_zlx)

    Parameters
    ----------
    p_xyz : matrix
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        p_zlx
    """
    p_xz = tr_y(p_xyz, dx, dy, dz)
    proj = np.kron(lin.inv(sqrt(tr_yz(p_xyz, dx, dy, dz))), np.eye(dz))
    return np.matmul(np.matmul(proj, p_xz), proj)

def get_zly(p_xyz, dx, dy, dz):
    """Calculates the conditional state of z given y (p_zly)

    Parameters
    ----------
    p_xyz : matrix
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        p_zly
    """
    p_yz = tr_x(p_xyz, dx, dy, dz)
    proj = np.kron(lin.inv(sqrt(tr_xz(p_xyz, dx, dy, dz))), np.eye(dz))
    return np.matmul(np.matmul(proj, p_yz), proj)
    

In [144]:
def R_c(p_cz, dc, dz, log_reg):
    """Mixes p_cz with a uniform distribution on z by a factor of the log regularization parameter

    Parameters
    ----------
    p_cz : matrix
        The joint density matrix of systems c and z (c being either x or y)
    dc: int
        The dimension of system c (c being either x or y)
    dz: int
        The dimension of system z
    log_reg: float
        The log regularization parameter

    Returns
    -------
    matrix
        The resultant density operator
    """
    return ((1-log_reg) * p_cz) + (log_reg * np.kron(np.eye(dc), np.eye(dz)/dz))

def R_z(p_z, dz, log_reg):
    """Mixes p_z with a uniform distribution on z by a factor of the log regularization parameter

    Parameters
    ----------
    p_z : matrix
        The density matrix of system z
    dz: int
        The dimension of system z
    log_reg: float
        The log regularization parameter

    Returns
    -------
    matrix
        The resultant density operator
    """
    return ((1-log_reg) * p_z) + (log_reg/dz * np.eye(dz))

def L_x(A, dx, dy, dz):
    """Lifts a state z|x to xyz by tensoring with the identity on y

    Parameters
    ----------
    A : matrix
        The state z|x
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        The original state A tensored with the identity on the y system
    """
    A = np.kron(np.eye(dy), A)

    # Swap yxz to xyz
    A_1 = np.zeros(np.shape(A))
    for x in range(dx):
        for y in range(dy):
            for z in range(dz):
                for i in range(dx*dy*dz):
                    A_1[x*(dy*dz) + y*dz + z][i] = A[x*dz + y*(dx*dz) + z][i]
    
    A_2 = np.zeros(np.shape(A))
    for x in range(dx):
        for y in range(dy):
            for z in range(dz):
                for i in range(dx*dy*dz):
                    A_2[i][x*(dy*dz) + y*dz + z] = A_1[i][x*dz + y*(dx*dz) + z]

    return A_2

def L_y(A, dx):
    """Lifts a state z|y to xyz by tensoring with the identity on x

    Parameters
    ----------
    A : matrix
        The state z|y
    dx: int
        The dimension of system x

    Returns
    -------
    matrix
        The original state A tensored with the identity on the x system
    """
    return np.kron(np.eye(dx), A)

def L_z(A, dx, dy):
    """Lifts a state z to xyz by tensoring with the identity on x and y

    Parameters
    ----------
    A : matrix
        The state z
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y

    Returns
    -------
    matrix
        The original state A tensored with the identity on the x and y systems
    """
    return np.kron(np.eye(dx*dy), A)

def condit_norm(R, dx, dy, dz):
    """Calculates the conditional state of system z given systems x and y

    Parameters
    ----------
    R : matrix
        The joint density matrix of systems x, y, and z
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        The resultant conditional density operator z|xy
    """
    M_r = tr_z(R, dx, dy, dz)
    proj = np.kron(np.inv(sqrt(M_r)), np.eye(dz))
    return np.matmul(np.matmul(proj, R), proj)

In [ ]:
def vn_entropy(p_c):
    """Calculates von Neumann entropy of system c

    Parameters
    ----------
    p_c : matrix
        The density matrix of system c

    Returns
    -------
    float
        The von Neumann entropy of system c
    """
    eigvals = lin.eig(p_c)[0]
    vn = 0
    for val in eigvals:
        if val != 0:
            vn += -1 * val * math.log(val, 2)
    
    return vn

def mi_xy(p_xy, dx, dy):
    """Calculates the mutual information of systems x and y

    Parameters
    ----------
    p_xy : matrix
        The joint density matrix of systems x and y
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y

    Returns
    -------
    float
        The mutual information of x and y
    """
    p_x = np.trace(p_xy.reshape(dx, dy, dx, dy), axis1=1, axis2=3)
    p_y = np.trace(p_xy.reshape(dx, dy, dx, dy), axis1=0, axis2=2)

    return vn_entropy(p_x) + vn_entropy(p_y) - vn_entropy(p_xy)

def mi_xylz(p_xyz, dx, dy, dz):
    """Calculates the conditional mutual information of systems x and y given system z

    Parameters
    ----------
    p_xyz : matrix
        The joint density matrix of systems x, y, and z
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    float
        The mutual information of x and y given z
    """
    vn_xz = vn_entropy(tr_y(p_xyz, dx, dy, dz))
    vn_yz = vn_entropy(tr_x(p_xyz, dx, dy, dz))
    vn_z = vn_entropy(tr_xy(p_xyz, dx, dy, dz))
    vn_xyz = vn_entropy(p_xyz)

    return vn_xz + vn_yz - vn_z - vn_xyz

In [148]:
# Testing

# A = np.array([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]]).reshape(4, 4)

# print(A)
#print(A.reshape(2, 2, 2, 2))
#print(np.trace(A.reshape(2,2,2,2), axis1=1, axis2=3))
# print(np.trace(A.reshape(2,2,2,2), axis1=0, axis2=1))

# psi = qt.tensor(qt.basis(2, 0), qt.basis(2, 1), (qt.basis(2, 0) + qt.basis(2, 1)).unit())
# x = np.kron(np.kron(np.array([[1, 0], [0, 0]]), np.array([[0, 0], [0, 1]])), np.array([[0.5, 0.5], [0.5, 0.5]]))

# print(psi.ptrace([0]))
# print(tr_yz(x, 2, 2, 2))

# print(np.exp(np.array([0, 1, 2])))

# A = np.kron(np.kron(np.array([[0.4, 0.1], [0.6, 0.7]]), np.array([[1, 0], [0, 1]])), np.array([[0.2, 1.9], [3.5, 0]]))
# A_1 = np.kron(np.array([[0.4, 0.1], [0.6, 0.7]]), np.array([[0.2, 1.9], [3.5, 0]]))
# print(A)
# print(L_x(A_1, 2, 2, 2))
# print(np.array_equal(A, L_x(A_1, 2, 2, 2)))

print(lin.eig(np.array([[0.5, 800], [234, 0.5]])))


EigResult(eigenvalues=array([ 433.16615306, -432.16615306]), eigenvectors=array([[ 0.87959899, -0.87959899],
       [ 0.47571589,  0.47571589]]))


In [ ]:
def initC(dx, dy, dz):
    """Generates a pseudo random positive operator C such that C is positive semidefinite and
    tracing out system z results in the identity over systems x and y

    Parameters
    ----------
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The dimension of system z

    Returns
    -------
    matrix
        A pseudo random conditional density matrix
    """
    # Generate random matrix
    A = np.random.rand(dx*dy*dz, dx*dy*dz)
    # Make A a positive matrix
    A = np.matmul(A, A)
    # Give A trace 1
    tr_A = np.trace(A)
    A = A/tr_A

    #Condition on X and Y
    p_xy = tr_z(A, dx, dy, dz)
    proj = np.kron(np.inv(sqrt(p_xy)), np.eye(dz))

    return np.matmul(np.matmul(proj, A), proj)

In [ ]:
def QLatentSearch(esti_state, dx, dy, dz, smoothing, damping, log_reg, penalty, n):
    """Heuristically searches for a stable point of the trade-off between the von Neumann
    entropy of the hidden common cause and mutual information between the two observed systems

    Parameters
    ----------
    esti_state: matrix
        The joint density matrix for the estimated state of the observed systems
    dx: int
        The dimension of system x
    dy: int
        The dimension of system y
    dz: int
        The maximum allowed dimension of system z (hidden common cause)
    smoothing: float
        The factor to which the estimated state should be smoothed in order to stabilize
        the density matrix
    damping: float
        The factor to which the newly calculated iteration of C_z|xy contributes to the updated
        version of C_z|xy
    log_reg: float
        The log regularization parameter
    penalty: float
        The factor on the von Neumann entropy of the hidden common cause in the objective
        function
    n: int
        The number of iterations that the algorithm will perform to find a stable point

    Returns
    -------
    (matrix, float, float)
        - A proposed joint density matrix for systems x, y, and z denoting an estimated stable
        point on the objective function
        - The conditional mutual information of systems x and y given system z for the proposed
        joint state
        - The von Neumann entropy of the proposed system z
    """
    smooth_esti = ((1-smoothing) * esti_state) + ((smoothing/(dx*dy)) * np.eye(dx*dy))
    C_zlxy = initC(dx, dy, dz)
    for i in range(n):
        #Form p_xyz
        condit_projector = np.kron(sqrt(smooth_esti), np.eye(dz))
        p_xyz = np.matmul(np.matmul(condit_projector, C_zlxy), condit_projector)

        #Compute C_zlx, C_zly, & p_z
        C_zlx = get_zlx(p_xyz, dx, dy, dz)
        C_zly = get_zly(p_xyz, dx, dy, dz)
        p_z = tr_xy(p_xyz, dx, dy, dz)

        #Compute R
        Lx = L_x(log(R_c(C_zlx, dx, dz, log_reg)))
        Ly = L_y(log(R_c(C_zly, dy, dz, log_reg)))
        Lz = L_z(log(R_z(p_z, dz, log_reg)))
        R_xyz = exp(Lx + Ly + (penalty-1)*Lz)

        #Compute C~ and C
        C_norm = condit_norm(R_xyz)
        C_zlxy = ((1-damping)*C_zlxy) + (damping * C_norm)

    #Form p_xyz
    condit_projector = np.kron(sqrt(smooth_esti), np.eye(dz))
    p_xyz = np.matmul(np.matmul(condit_projector, C_zlxy), condit_projector)
    
    return p_xyz, mi_xylz(p_xyz, dx, dy, dz), vn_entropy(tr_xy(p_xyz, dx, dy, dz))

In [ ]:
def QInferGraph(esti_state, dx, dy, dz, penalties, tolerance, entrop_thresh, extern_thresh, dep_gate, smoothing, damping, log_reg, n):
    smooth_esti = ((1-smoothing) * esti_state) + ((smoothing/(dx*dy)) * np.eye(dx*dy))
    if mi_xy(smooth_esti, dx, dy) <= dep_gate:
        return "not latent (too little dependence)"
    
    for penalty in penalties:
        p_xyz, mi, sz = QLatentSearch(esti_state, dx, dy, dz, smoothing, damping, log_reg, penalty, n)
    
    
